# Urban Anomaly Detection Network Demo

This notebook demonstrates the Urban Anomaly Detection Network for learning normal traffic patterns and detecting anomalies in urban surveillance videos.

## Overview

The system uses:
- **Spatio-Temporal Autoencoder**: ConvLSTM-based architecture for unsupervised learning
- **Reconstruction Error**: Anomalies detected as deviations from normal patterns
- **Scalable Processing**: Batch processing for real-world deployment

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from models.autoencoder import SpatioTemporalAutoencoder
from models.anomaly_detector import AnomalyDetector
from utils.data_utils import SyntheticTrafficDataset, create_data_loaders
from utils.visualization import visualize_reconstruction, visualize_anomaly_detection

%matplotlib inline

## 1. Create Synthetic Traffic Data

For demonstration, we'll use synthetic traffic patterns. In production, replace this with real surveillance videos.

In [ ]:
# Configuration
SEQUENCE_LENGTH = 16
FRAME_SIZE = (128, 128)
BATCH_SIZE = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")

# Create synthetic dataset
train_dataset = SyntheticTrafficDataset(
    num_samples=80,
    sequence_length=SEQUENCE_LENGTH,
    frame_size=FRAME_SIZE
)

val_dataset = SyntheticTrafficDataset(
    num_samples=20,
    sequence_length=SEQUENCE_LENGTH,
    frame_size=FRAME_SIZE
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Visualize sample sequence
sample = train_dataset[0]
print(f"Sequence shape: {sample.shape}")  # (T, C, H, W)

# Display sample frames
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    frame_idx = i * (SEQUENCE_LENGTH // 4)
    frame = sample[frame_idx].permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'Frame {frame_idx}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 2. Create and Train the Model

The Spatio-Temporal Autoencoder learns normal traffic patterns in an unsupervised manner.

In [ ]:
# Create model
model = SpatioTemporalAutoencoder(
    input_channels=3,
    hidden_channels=[32, 64, 128],  # Smaller for demo
    kernel_size=3
).to(DEVICE)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")

In [ ]:
# Training setup
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 10  # Short training for demo

In [ ]:
# Training loop
train_losses = []
val_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0
    
    for videos in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}'):
        videos = videos.to(DEVICE)
        
        # Forward pass
        reconstructed = model(videos)
        loss = criterion(reconstructed, videos)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validate
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for videos in val_loader:
            videos = videos.to(DEVICE)
            reconstructed = model(videos)
            loss = criterion(reconstructed, videos)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")

print("\nTraining completed!")

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Visualize Reconstruction

Compare original and reconstructed frames to verify the model learned the patterns.

In [ ]:
# Get sample for reconstruction
model.eval()
sample_video = val_dataset[0].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    reconstructed_video = model(sample_video)

# Visualize
fig = visualize_reconstruction(sample_video[0], reconstructed_video[0], num_frames=8)
plt.show()

## 4. Anomaly Detection

Use reconstruction error to detect anomalous traffic patterns.

In [ ]:
# Create anomaly detector
detector = AnomalyDetector(model, threshold_percentile=95)

# Fit threshold on normal data
threshold = detector.fit_threshold(train_loader, device=DEVICE)
print(f"\nAnomaly threshold: {threshold:.6f}")

In [ ]:
# Test anomaly detection
test_videos = torch.stack([val_dataset[i] for i in range(5)]).to(DEVICE)

is_anomaly, errors = detector.detect_anomalies(
    test_videos, device=DEVICE, return_errors=True
)

print("\nAnomaly Detection Results:")
print(f"Threshold: {detector.threshold:.6f}")
for i, (anomaly, error) in enumerate(zip(is_anomaly, errors)):
    status = "ANOMALY" if anomaly else "NORMAL"
    print(f"Video {i}: {status:8s} (error: {error:.6f})")

In [ ]:
# Visualize anomaly detection with spatial heatmap
sample_idx = 0
sample_video = test_videos[sample_idx:sample_idx+1]

# Get spatial anomaly map
anomaly_map = detector.get_spatial_anomaly_map(sample_video, device=DEVICE)

# Visualize
fig = visualize_anomaly_detection(
    test_videos[sample_idx],
    anomaly_map[0],
    is_anomaly[sample_idx].item()
)
plt.show()

## 5. Summary

This notebook demonstrated:

1. **Unsupervised Learning**: Training on normal traffic without manual annotations
2. **Spatio-Temporal Modeling**: ConvLSTM autoencoder for video understanding
3. **Anomaly Detection**: Identifying deviations from learned patterns
4. **Visualization**: Spatial heatmaps showing anomalous regions

### Next Steps

- Train on real surveillance videos for production deployment
- Fine-tune hyperparameters for specific scenarios
- Integrate with real-time video processing pipeline
- Add alert system for detected anomalies

In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'threshold': detector.threshold,
    'config': {
        'hidden_channels': [32, 64, 128],
        'kernel_size': 3,
        'sequence_length': SEQUENCE_LENGTH,
        'frame_size': FRAME_SIZE
    }
}, 'demo_model.pth')

print("Model saved to 'demo_model.pth'")